# Smart Fraud Detection Pipeline
## Silver Layer - Data Cleaning, Validation and Enrichment

The Silver layer transforms the raw Bronze Delta tables into cleaned, standardized, validated, and enriched datasets.

#### Objectives

- Standardize raw data
- Handle missing values
- Cast columns to appropriate data types
- Remove duplicate records
- Validate referential relationships
- Enrich transactions with account information
- Preserve data-quality issues using validation flags
- Store trusted datasets as Delta tables

#### Input Tables

- fraud_detection.bronze.accounts
- fraud_detection.bronze.transactions
- fraud_detection.bronze.known_fraud_accounts

#### Output Tables

- fraud_detection.silver.accounts
- fraud_detection.silver.transactions
- fraud_detection.silver.known_fraud_accounts
- fraud_detection.silver.enriched_transactions

In [0]:
# import
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# configuration
CATALOG = "fraud_detection"

BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

BRONZE_ACCOUNTS = f"{BRONZE_SCHEMA}.accounts"
BRONZE_TRANSACTIONS = f"{BRONZE_SCHEMA}.transactions"
BRONZE_FRAUD = f"{BRONZE_SCHEMA}.known_fraud_accounts"

SILVER_ACCOUNTS = f"{SILVER_SCHEMA}.accounts"
SILVER_TRANSACTIONS = f"{SILVER_SCHEMA}.transactions"
SILVER_FRAUD = f"{SILVER_SCHEMA}.known_fraud_accounts"
SILVER_ENRICHED = f"{SILVER_SCHEMA}.enriched_transactions"

**Read bronze tables**

In [0]:
df_accounts_raw = spark.table(BRONZE_ACCOUNTS)
df_transactions_raw = spark.table(BRONZE_TRANSACTIONS)
df_fraud_raw = spark.table(BRONZE_FRAUD)

print("Bronze tables loaded successfully.")

Bronze tables loaded successfully.


In [0]:
print("Accounts:", df_accounts_raw.count())
print("Transactions:", df_transactions_raw.count())
print("Fraud Watchlist:", df_fraud_raw.count())

Accounts: 50
Transactions: 200
Fraud Watchlist: 10


**Clean Accounts**

The raw Bronze account fields are strings so we are creating proper data types.

In [0]:
df_accounts = (
    df_accounts_raw
    .withColumn("account_id", F.trim(F.col("account_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("account_type", F.upper(F.trim(F.col("account_type"))))
    .withColumn("branch", F.trim(F.col("branch")))
    .withColumn("kyc_status", F.upper(F.trim(F.col("kyc_status"))))
    .withColumn("opening_date",F.to_date(F.col("opening_date")))
    .withColumn("credit_limit",F.col("credit_limit").cast("double"))
)

In [0]:
# handle account nulls
df_accounts = (
    df_accounts
    .withColumn(
        "customer_name",
        F.coalesce(F.col("customer_name"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "account_type",
        F.coalesce(F.col("account_type"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "branch",
        F.coalesce(F.col("branch"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "kyc_status",
        F.coalesce(F.col("kyc_status"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "credit_limit_missing",
        F.col("credit_limit").isNull()
    )
)


In [0]:
# remove duplicates accounts
duplicate_accounts = (
    df_accounts
    .groupBy("account_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_accounts.isEmpty():
    print("No duplicates")
else:
    display(duplicate_accounts)

No duplicates


**Clean transactions**

In [0]:
# clean transactions
df_transactions = (
    df_transactions_raw
    .withColumn("txn_id", F.trim(F.col("txn_id")))
    .withColumn("account_id", F.trim(F.col("account_id")))
    .withColumn("txn_type", F.upper(F.trim(F.col("txn_type"))))
    .withColumn("merchant", F.trim(F.col("merchant")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("txn_date",F.to_date(F.col("txn_date")))
    .withColumn("amount",F.col("amount").cast("double")
    )
)

In [0]:
# standardize international flag
df_transactions = (
    df_transactions
    .withColumn(
        "is_international",
        F.when(
            F.upper(F.trim(F.col("is_international"))).isin(
                "TRUE", "YES", "Y", "1"
            ),
            F.lit(True)
        )
        .when(
            F.upper(F.trim(F.col("is_international"))).isin(
                "FALSE", "NO", "N", "0"
            ),
            F.lit(False)
        )
        .otherwise(None)
    )
)

In [0]:
# handle missing values
df_transactions = (
    df_transactions
    .withColumn(
        "merchant",
        F.coalesce(F.col("merchant"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "city",
        F.coalesce(F.col("city"), F.lit("UNKNOWN"))
    )
    .withColumn(
        "txn_type",
        F.coalesce(F.col("txn_type"), F.lit("UNKNOWN"))
    )
)

In [0]:
df_transactions = df_transactions.withColumn(
    "amount_missing",
    F.col("amount").isNull()
)

In [0]:
duplicate_transactions = (
    df_transactions
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_transactions.isEmpty():
    print("No duplicates")
else:
    display(duplicate_transactions)

No duplicates


In [0]:
# validate transaction amounts
invalid_amounts = (
    df_transactions
    .filter(
        F.col("amount").isNull() |
        (F.col("amount") <= 0)
    )
)

print("Transactions with missing or non-positive amounts:",
    invalid_amounts.count()
)

Transactions with missing or non-positive amounts: 0


In [0]:
# add amount validation flag
# these will allow us to decide whether these transactions should be consider for financial metrics
df_transactions = (
    df_transactions
    .withColumn(
        "amount_valid",
        F.when(
            F.col("amount").isNull() |
            (F.col("amount") <= 0),
            F.lit(False)
        )
        .otherwise(F.lit(True))
    )
)

**Clean fraud watchlist**

In [0]:
df_fraud = (
    df_fraud_raw
    .withColumn(
        "account_id",
        F.trim(F.col("account_id"))
    )
    .withColumn(
        "fraud_type",
        F.upper(F.trim(F.col("fraud_type")))
    )
    .withColumn(
        "flagged_date",
        F.to_date(F.col("flagged_date"))
    )
)

In [0]:
# handle missing values
df_fraud = df_fraud.withColumn(
    "fraud_type",
    F.coalesce(F.col("fraud_type"), F.lit("UNKNOWN"))
)

In [0]:
# duplicate validation
# Check for duplicates
duplicate_fraud = (
    df_fraud
    .groupBy("account_id", "fraud_type", "flagged_date")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_fraud.isEmpty():
    print("No duplicates found")
else:
    print("Duplicates found:")
    display(duplicate_fraud)

# Remove duplicates
df_fraud = df_fraud.dropDuplicates(
    ["account_id", "fraud_type", "flagged_date"]
)

No duplicates found


Checking Account and Transaction relationship

In [0]:
account_reference = (
    df_accounts
    .select("account_id")
    .dropDuplicates()
    .withColumn("account_exists", F.lit(True))
)

In [0]:
df_transactions_validated = (
    df_transactions
    .join(
        account_reference,
        on="account_id",
        how="left"
    )
    .withColumn(
        "account_exists",
        F.coalesce(
            F.col("account_exists"),
            F.lit(False)
        )
    )
)

In [0]:
# measure orphan transactions 
# account_id which are not present in account table
orphan_transactions = (
    df_transactions_validated
    .filter(~F.col("account_exists"))
)

print("Transactions with no matching account:",orphan_transactions.count())


Transactions with no matching account: 8


In [0]:
display(orphan_transactions)

account_id,txn_id,txn_date,txn_type,amount,merchant,city,is_international,_source_file,_ingested_at,amount_missing,amount_valid,account_exists
ACC-00056,TXN-000001,2026-01-09,DEBIT,915931.22,UNKNOWN,Singapore,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false
ACC-00051,TXN-000026,2026-01-19,TRANSFER,1627.39,BookMyShow,Bangalore,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false
ACC-00054,TXN-000051,2026-02-11,PAYMENT,5599.04,Crypto_Exchange,Chennai,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false
ACC-00058,TXN-000076,2026-03-19,PAYMENT,21055.6,Airtel,New York,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false
ACC-00059,TXN-000101,2026-02-24,CREDIT,540260.02,Amazon,Bangalore,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false
ACC-00053,TXN-000126,2026-03-28,PAYMENT,6083.47,MakeMyTrip,Delhi,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false
ACC-00053,TXN-000151,2026-02-16,CREDIT,7845.87,PhonePe,New York,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false
ACC-00055,TXN-000176,2026-02-27,CREDIT,6448.34,MakeMyTrip,Lagos,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,false


In [0]:
# create enriched transactions
account_details = (
    df_accounts
    .select(
        "account_id",
        "customer_name",
        "account_type",
        "opening_date",
        "branch",
        "kyc_status",
        "credit_limit"
    )
)

In [0]:
df_enriched_transactions = (
    df_transactions
    .join(
        account_details,
        on="account_id",
        how="left"
    )
)

In [0]:
display(df_enriched_transactions)

account_id,txn_id,txn_date,txn_type,amount,merchant,city,is_international,_source_file,_ingested_at,amount_missing,amount_valid,customer_name,account_type,opening_date,branch,kyc_status,credit_limit
ACC-00056,TXN-000001,2026-01-09,DEBIT,915931.22,UNKNOWN,Singapore,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,null,null,null,null,null,null
ACC-00025,TXN-000002,2026-01-11,TRANSFER,23892.36,BookMyShow,Bangalore,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Hemant Jain,NRI,2023-02-17,Hyderabad_Hitech,VERIFIED,500000.0
ACC-00029,TXN-000003,2026-02-03,CREDIT,15277.03,Swiggy,New York,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Tanuja Nair,SALARY,2024-05-02,Pune_FC,VERIFIED,100000.0
ACC-00007,TXN-000004,2026-03-05,WITHDRAWAL,5498.74,ATM,Delhi,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Prakash Rao,NRI,2016-10-04,Pune_FC,EXPIRED,500000.0
ACC-00029,TXN-000005,2026-02-23,DEBIT,919.78,Flipkart,Chennai,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Tanuja Nair,SALARY,2024-05-02,Pune_FC,VERIFIED,100000.0
ACC-00047,TXN-000006,2026-02-09,DEBIT,22417.98,Swiggy,Mumbai,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Pawan Arora,SAVINGS,2024-02-02,Hyderabad_Hitech,PENDING,100000.0
ACC-00003,TXN-000007,2026-02-22,TRANSFER,3832.61,BigBasket,Singapore,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Anil Reddy,CURRENT,2022-07-03,Bangalore_MG,VERIFIED,null
ACC-00037,TXN-000008,2026-03-01,WITHDRAWAL,11979.58,PhonePe,Delhi,false,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Naveen Chadha,SALARY,2015-11-09,Kolkata_Park,EXPIRED,50000.0
ACC-00035,TXN-000009,2026-01-08,WITHDRAWAL,23856.18,Unknown_Merchant,New York,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Tarun Bose,SAVINGS,2017-10-27,Delhi_CP,EXPIRED,500000.0
ACC-00008,TXN-000010,2026-03-19,TRANSFER,8369.32,BookMyShow,Singapore,true,dbfs:/Volumes/fraud_detection/source/raw_files/transactions.csv,2026-08-08T14:39:19.087Z,false,true,Lata Mishra,SALARY,2022-11-22,Mumbai_Main,VERIFIED,100000.0


In [0]:
# add account existence flag to enriched data
account_ids = (
    df_accounts
    .select("account_id")
    .distinct()
    .withColumn("_account_exists", F.lit(True))
)

df_enriched_transactions = (
    df_transactions
    .join(
        account_details,
        on="account_id",
        how="left"
    )
    .join(
        account_ids,
        on="account_id",
        how="left"
    )
    .withColumn(
        "account_exists",
        F.coalesce(
            F.col("_account_exists"),
            F.lit(False)
        )
    )
    .drop("_account_exists")
)

In [0]:
# final check
account_nulls = df_accounts.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_accounts.columns
])

display(account_nulls)


account_id,customer_name,account_type,opening_date,branch,kyc_status,credit_limit,_source_file,_ingested_at,credit_limit_missing
0,0,0,0,0,0,9,0,0,0


In [0]:
transaction_nulls = df_transactions.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_transactions.columns
])

display(transaction_nulls)

txn_id,account_id,txn_date,txn_type,amount,merchant,city,is_international,_source_file,_ingested_at,amount_missing,amount_valid
0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# validation summary
silver_quality_summary = {
    "accounts_total": df_accounts.count(),
    "transactions_total": df_transactions.count(),
    "fraud_watchlist_total": df_fraud.count(),

    "duplicate_accounts": duplicate_accounts.count(),
    "duplicate_transactions": duplicate_transactions.count(),

    "missing_credit_limits": (
        df_accounts
        .filter(F.col("credit_limit").isNull())
        .count()
    ),

    "missing_transaction_amounts": (
        df_transactions
        .filter(F.col("amount").isNull())
        .count()
    ),

    "invalid_transaction_amounts": (
        df_transactions
        .filter(~F.col("amount_valid"))
        .count()
    ),

    "orphan_transactions": (
        df_enriched_transactions
        .filter(~F.col("account_exists"))
        .count()
    )
}

for metric, value in silver_quality_summary.items():
    print(f"{metric:35} : {value}")

accounts_total                      : 50
transactions_total                  : 200
fraud_watchlist_total               : 10
duplicate_accounts                  : 0
duplicate_transactions              : 0
missing_credit_limits               : 9
missing_transaction_amounts         : 0
invalid_transaction_amounts         : 0
orphan_transactions                 : 8


**Save Silver tables**

In [0]:
# save silver accounts
(
    df_accounts
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_ACCOUNTS)
)

In [0]:
# save silver transactions
(
    df_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TRANSACTIONS)
)

In [0]:
# save silver fraud watchlist
(
    df_fraud
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FRAUD)
)

In [0]:
# save enriched transactions
(
    df_enriched_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_ENRICHED)
)

In [0]:
# verify tables
spark.sql("""
SHOW TABLES IN fraud_detection.silver
""").show(truncate=False)


+--------+---------------------+-----------+
|database|tableName            |isTemporary|
+--------+---------------------+-----------+
|silver  |accounts             |false      |
|silver  |enriched_transactions|false      |
|silver  |known_fraud_accounts |false      |
|silver  |transactions         |false      |
+--------+---------------------+-----------+

